#### Импорты, seed и устройство

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import json
import csv
import os

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


#### Данные и DataLoader

In [2]:
transform = transforms.ToTensor()

def get_emnist_dataloaders(batch_size=128, val_ratio=0.2, seed=SEED):
    train_full = datasets.EMNIST(
        root="./data",
        split="balanced",
        train=True,
        download=True,
        transform=transform
    )
    test = datasets.EMNIST(
        root="./data",
        split="balanced",
        train=False,
        download=True,
        transform=transform
    )

    val_size = int(len(train_full) * val_ratio)
    train_size = len(train_full) - val_size

    train_ds, val_ds = random_split(
        train_full,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seed)
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_emnist_dataloaders()


#### MLP-модель

In [3]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(512, 256), dropout_p=0.0, use_bn=False):
        super().__init__()

        layers = [nn.Flatten()] 

        in_features = 28 * 28
        for h in hidden_sizes:
            layers.append(nn.Linear(in_features, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            in_features = h

        layers.append(nn.Linear(in_features, 47)) 

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


Accuracy

In [4]:
def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

Train и Evaluate

In [5]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


#### Часть A (S08)

Функция экперимента

In [6]:
def run_experiment(
    exp_id,
    dropout=0.0,
    use_bn=False,
    optimizer_name="Adam",
    lr=1e-3,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=20,
    early_stopping=False,
    patience=5,
):
    print(f"\n=== {exp_id} ===")
    print(f"dropout={dropout}, bn={use_bn}, opt={optimizer_name}, lr={lr}, "
          f"momentum={momentum}, wd={weight_decay}, early_stopping={early_stopping}")

    model = MLP(dropout_p=dropout, use_bn=use_bn).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    best_val_acc = 0.0
    best_state = None
    wait = 0

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:02d}: "
              f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
              f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if early_stopping and wait >= patience:
            print(f"Early stopping at epoch {epoch} (patience={patience})")
            break

    return best_val_acc, best_state, history


E1–E4

In [7]:
results = {}

results["E1"] = run_experiment("E1", dropout=0.0, use_bn=False)
results["E2"] = run_experiment("E2", dropout=0.3, use_bn=False)
results["E3"] = run_experiment("E3", dropout=0.0, use_bn=True)

best_reg_exp = max(["E2", "E3"], key=lambda e: results[e][0])
print("Лучший эксперимент регуляризации:", best_reg_exp)

if best_reg_exp == "E2":
    results["E4"] = run_experiment("E4", dropout=0.3, use_bn=False, early_stopping=True)
else:
    results["E4"] = run_experiment("E4", dropout=0.0, use_bn=True, early_stopping=True)

best_model_state = results["E4"][1]
torch.save(best_model_state, "artifacts/best_model.pt")



=== E1 ===
dropout=0.0, bn=False, opt=Adam, lr=0.001, momentum=0.0, wd=0.0, early_stopping=False
Epoch 01: train_loss=1.0925, val_loss=0.6929, train_acc=0.6799, val_acc=0.7824
Epoch 02: train_loss=0.5838, val_loss=0.5482, train_acc=0.8097, val_acc=0.8219
Epoch 03: train_loss=0.4770, val_loss=0.5028, train_acc=0.8383, val_acc=0.8320
Epoch 04: train_loss=0.4159, val_loss=0.4811, train_acc=0.8556, val_acc=0.8396
Epoch 05: train_loss=0.3724, val_loss=0.4641, train_acc=0.8667, val_acc=0.8437
Epoch 06: train_loss=0.3384, val_loss=0.4626, train_acc=0.8756, val_acc=0.8468
Epoch 07: train_loss=0.3082, val_loss=0.4586, train_acc=0.8854, val_acc=0.8504
Epoch 08: train_loss=0.2846, val_loss=0.4714, train_acc=0.8921, val_acc=0.8474
Epoch 09: train_loss=0.2623, val_loss=0.4874, train_acc=0.8992, val_acc=0.8467
Epoch 10: train_loss=0.2409, val_loss=0.4807, train_acc=0.9049, val_acc=0.8518
Epoch 11: train_loss=0.2256, val_loss=0.5033, train_acc=0.9101, val_acc=0.8483
Epoch 12: train_loss=0.2104, val_

Часть B (S09)

In [8]:
results["O1"] = run_experiment(
    "O1",
    dropout=0.3 if best_reg_exp == "E2" else 0.0,
    use_bn=(best_reg_exp == "E3"),
    optimizer_name="Adam",
    lr=1e-1,
    max_epochs=8,
)

results["O2"] = run_experiment(
    "O2",
    dropout=0.3 if best_reg_exp == "E2" else 0.0,
    use_bn=(best_reg_exp == "E3"),
    optimizer_name="Adam",
    lr=1e-5,
    max_epochs=8,
)

results["O3"] = run_experiment(
    "O3",
    dropout=0.3 if best_reg_exp == "E2" else 0.0,
    use_bn=(best_reg_exp == "E3"),
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
    max_epochs=15,
)



=== O1 ===
dropout=0.3, bn=False, opt=Adam, lr=0.1, momentum=0.0, wd=0.0, early_stopping=False
Epoch 01: train_loss=4.7200, val_loss=3.8589, train_acc=0.0242, val_acc=0.0215
Epoch 02: train_loss=3.8766, val_loss=3.8616, train_acc=0.0212, val_acc=0.0207
Epoch 03: train_loss=3.8669, val_loss=3.8630, train_acc=0.0215, val_acc=0.0208
Epoch 04: train_loss=3.8646, val_loss=3.8627, train_acc=0.0209, val_acc=0.0207
Epoch 05: train_loss=3.8650, val_loss=3.8674, train_acc=0.0208, val_acc=0.0210
Epoch 06: train_loss=3.8649, val_loss=3.8646, train_acc=0.0218, val_acc=0.0210
Epoch 07: train_loss=3.8652, val_loss=3.8628, train_acc=0.0206, val_acc=0.0210
Epoch 08: train_loss=3.8639, val_loss=3.8628, train_acc=0.0212, val_acc=0.0227

=== O2 ===
dropout=0.3, bn=False, opt=Adam, lr=1e-05, momentum=0.0, wd=0.0, early_stopping=False
Epoch 01: train_loss=3.7729, val_loss=3.6121, train_acc=0.0960, val_acc=0.2705
Epoch 02: train_loss=3.3630, val_loss=3.0147, train_acc=0.2348, val_acc=0.3776
Epoch 03: train_

#### Артефакты

In [9]:
best_config = {
    "dataset": "EMNIST Balanced",
    "seed": SEED,
    "model": "MLP(28*28 → 512 → 256 → 47)",
    "dropout": 0.3 if best_reg_exp == "E2" else 0.0,
    "batchnorm": bool(best_reg_exp == "E3"),
    "optimizer": "Adam",
    "lr": 1e-3,
    "early_stopping": True,
    "patience": 5,
}

with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config, f, indent=4)


In [10]:
with open("artifacts/runs.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "experiment_id", "dataset", "seed", "model_summary",
        "optimizer", "lr", "momentum", "weight_decay",
        "epochs_trained", "best_val_accuracy",
    ])

    for exp_id, (val_acc, _, hist) in results.items():
        if exp_id.startswith("E"):
            opt = "Adam"; lr = 1e-3; momentum = 0.0; wd = 0.0
        elif exp_id == "O1":
            opt = "Adam"; lr = 1e-1; momentum = 0.0; wd = 0.0
        elif exp_id == "O2":
            opt = "Adam"; lr = 1e-5; momentum = 0.0; wd = 0.0
        else:
            opt = "SGD"; lr = 1e-2; momentum = 0.9; wd = 1e-4

        writer.writerow([
            exp_id,
            "EMNIST Balanced",
            SEED,
            "MLP 512-256-47",
            opt,
            lr,
            momentum,
            wd,
            len(hist["train_loss"]),
            val_acc,
        ])


Графики

In [11]:
def plot_history(history, title, filename):
    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.plot(history["train_loss"], label="train")
    plt.plot(history["val_loss"], label="val")
    plt.title("Loss")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history["train_acc"], label="train")
    plt.plot(history["val_acc"], label="val")
    plt.title("Accuracy")
    plt.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

plot_history(results["E4"][2], "Best run (E4)", "artifacts/figures/curves_best.png")
plot_history(results["O1"][2], "LR too big (O1)", "artifacts/figures/curves_lr_extremes_O1.png")
plot_history(results["O2"][2], "LR too small (O2)", "artifacts/figures/curves_lr_extremes_O2.png")
